In [8]:
from classes.ml_recruitment import ML_Recruitment
import os
from numpy import nan

vCaminhoBase = os.path.join("..", 'etl', "output", "dataset_unificado_balanceado.csv")

vML = ML_Recruitment(pCaminhoBase=vCaminhoBase)

# Carregar a base de dados
vML.carregarBase()

# Criar uma coluna de teste com valores NaN
vML.baseDeDados['TesTE de Nome de ColunA'] = nan

# Padronizar a base de dados
vML.padronizarBase(pListaDeColunasParaFillNA=['requisitos_vaga', 'cv_texto', 'TesTE de Nome de ColunA'], pPreencherNulosCom='vazio')

# Treinar o vetorizador textual
vML.treinarVetorizadorTextual(pListaColunas=['requisitos_vaga', 'cv_texto'], pNumeroMaximoFeatures=300)

# Calcular a similaridade textual
vML.baseDeDados['sim_textual'] = vML.calcularSimilaridadeTextual(pListaColunas=['requisitos_vaga', 'cv_texto'])

# Inserir novas colunas
vML.baseDeDados['match_nivel'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_ingles'] = (vML.baseDeDados['nivel_ingles_vaga'] == vML.baseDeDados['nivel_ingles_candidato']).astype(int)
vML.baseDeDados['match_profissional'] = (vML.baseDeDados['nivel_profissional_vaga'] == vML.baseDeDados['nivel_profissional_candidato']).astype(int)
vML.baseDeDados['match_espanhol'] = (vML.baseDeDados['nivel_espanhol_vaga'] == vML.baseDeDados['nivel_espanhol_candidato']).astype(int)
vML.baseDeDados['match_local'] = (vML.baseDeDados['local_vaga'] == vML.baseDeDados['local_candidato']).astype(int)
vML.baseDeDados['match_academico'] = (vML.baseDeDados['nivel_academico_vaga'] == vML.baseDeDados['nivel_academico_candidato']).astype(int)

# Separar features e target
vML.separarFeatureTarget(pColunaTarget='match', pColunasIgnorar=['situacao', 'comentario', 'recrutador', 'vaga_id', 'codigo_candidato', 'nome_candidato', 'conhecimentos_tecnicos', 'teste_de_nome_de_coluna'])

# Criar a pipeline de pré-processamento
vML.criarPipeline(
    pColunasNumericas=['sim_textual', 'match_nivel', 'match_ingles', 'match_profissional', 'match_espanhol', 'match_local', 'match_academico'],
    pColunasCategoricas=['titulo_vaga', 'nivel_profissional_vaga', 'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga', 'nivel_academico_candidato', 'nivel_ingles_candidato', 'nivel_espanhol_candidato', 'nivel_profissional_candidato', 'local_candidato', 'cliente', 'local_vaga'],
    pColunasTexto=['requisitos_vaga', 'cv_texto'],
    pNumeroMaximoFeatures=30
)

# Separar a base de dados em treino e teste
vML.separarTreinoTeste(pProporcaoTreino=0.2, pRandomState=42)

In [15]:
# vML.features_Treino
vML.pipeline.fit_transform(vML.features_Treino)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 204614 stored elements and shape (5108, 3600)>

In [10]:
vML.features.shape

(6386, 19)

In [5]:
a = vML.features.columns.to_list()

b = ['titulo_vaga', 'nivel_profissional_vaga', 'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga', 'nivel_academico_candidato', 'nivel_ingles_candidato', 'nivel_espanhol_candidato', 'nivel_profissional_candidato', 'local_candidato', 'cliente', 'local_vaga', 'sim_textual', 'match_nivel', 'match_ingles', 'match_profissional', 'match_espanhol', 'match_local', 'match_academico', 'requisitos_vaga', 'cv_texto']


In [7]:
[col for col in b if col not in a]

['requisitos_vaga', 'cv_texto']

In [10]:
print(vML)

Caminho da Base: ..\etl\output\dataset_unificado_balanceado.csv
Número de Linhas: 6386
Número de Colunas: 30
Tamanho das features: (6386, 21)
Tamanho do target: (6386,)
Número de Features de Treino: (5108, 21)
Número de Target de Treino: (5108,)
Número de Features de Teste: (1278, 21)
Número de Target de Teste: (1278,)
Pipeline: ColumnTransformer(sparse_threshold=1.0,
                  transformers=[('numerical',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['sim_textual', 'match_nivel', 'match_ingles',
                                  'match_profissional', 'match_espanhol',
                                  'match_local', 'match_academico']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='missing',
    

In [15]:
a = vML.features.columns.to_list()
a

['titulo_vaga',
 'cliente',
 'nivel_profissional_vaga',
 'nivel_academico_vaga',
 'nivel_ingles_vaga',
 'nivel_espanhol_vaga',
 'local_vaga',
 'nivel_academico_candidato',
 'nivel_ingles_candidato',
 'nivel_espanhol_candidato',
 'nivel_profissional_candidato',
 'local_candidato',
 'sim_textual',
 'match_nivel',
 'match_ingles',
 'match_profissional',
 'match_espanhol',
 'match_local',
 'match_academico']

In [9]:
cat_cols = [
    'titulo_vaga', 'nivel_profissional_vaga',
    'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga',
    'nivel_academico_candidato', 'nivel_ingles_candidato',
    'nivel_espanhol_candidato', 'nivel_profissional_candidato',
    'local_candidato', 'cliente', 'local_vaga'
]

num_cols = [
    'sim_textual', 'match_nivel', 'match_ingles',
    'match_profissional', 'match_espanhol', 'match_local',
    'match_academico'
]

vListaFinal = cat_cols + num_cols
vListaFinal

['titulo_vaga',
 'nivel_profissional_vaga',
 'nivel_ingles_vaga',
 'nivel_espanhol_vaga',
 'nivel_academico_vaga',
 'nivel_academico_candidato',
 'nivel_ingles_candidato',
 'nivel_espanhol_candidato',
 'nivel_profissional_candidato',
 'local_candidato',
 'cliente',
 'local_vaga',
 'sim_textual',
 'match_nivel',
 'match_ingles',
 'match_profissional',
 'match_espanhol',
 'match_local',
 'match_academico']

In [14]:
[a for a in vML.features.columns if a not in vListaFinal]

[]